# 05 · 自己实现 LLM 竞价 Agent

思路和平台一致：**每个时段调用一次 LLM**，给它看聚合状态（剩余预算、CPA 进度、市场价），
它回一个 JSON `{"action":"set_alpha","alpha":...}`；解析失败就 fallback。

需要：Bedrock 权限（跑一次 48 个调用，haiku 约 $0.05）。没有权限也可以先看 mock 模式。

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caitq2024/auto_auction/blob/main/notebooks/05_%E8%87%AA%E5%B7%B1%E5%AE%9E%E7%8E%B0LLM_Agent.ipynb)

> Colab 用户先运行下面的数据下载 cell；本地运行可跳过。


In [ ]:
# Colab 环境准备：拉取教学数据（本地运行且 data/ 已存在时自动跳过）
import os, urllib.request
os.makedirs('data', exist_ok=True)
for f in ['period7_adv0.csv.gz', 'period7_tick0_adv0to7.csv.gz']:
    if not os.path.exists(f'data/{f}'):
        urllib.request.urlretrieve(f'https://github.com/caitq2024/auto_auction/raw/main/notebooks/data/{f}', f'data/{f}')
        print('downloaded', f)


In [ ]:
import json, re, boto3
import pandas as pd, numpy as np

df = pd.read_csv('data/period7_adv0.csv.gz')
BUDGET, TARGET_CPA, NUM_TICK = float(df.budget.iloc[0]), float(df.CPAConstraint.iloc[0]), 48

PROMPT = '''You are an auto-bidding agent. Bid = alpha * pValue per impression.
Budget must last 48 steps; score = conversions, penalized by (target_cpa/cpa)^2 if cpa>target.
Unspent budget is pure loss — by step t you should have spent ~t/48 of budget.
Reply ONLY JSON: {"action":"set_alpha","alpha":<number>}

State: %s'''

In [ ]:
USE_REAL_LLM = False   # 有 Bedrock 权限改成 True

client = boto3.client('bedrock-runtime', region_name='us-west-2') if USE_REAL_LLM else None

def llm_alpha(state: dict, prev_alpha: float) -> float:
    if not USE_REAL_LLM:
        # mock：模拟一个"看 pacing 调价"的 LLM
        behind = state['remaining_ratio'] - state['time_ratio_remaining']
        return max(prev_alpha * (1.3 if behind > 0.05 else 0.8 if behind < -0.05 else 1.0), 10)
    resp = client.converse(
        modelId='us.anthropic.claude-haiku-4-5-20251001-v1:0',
        messages=[{'role':'user','content':[{'text': PROMPT % json.dumps(state)}]}],
        inferenceConfig={'maxTokens':200})
    text = next(b['text'] for b in resp['output']['message']['content'] if 'text' in b)
    m = re.search(r'\{[^{}]*\}', text)          # 提取首个 JSON 块
    try:
        alpha = float(json.loads(m.group())['alpha'])
        assert np.isfinite(alpha)
        return float(np.clip(alpha, 0, 2000))    # clip 到安全范围
    except Exception:
        return prev_alpha                         # fallback：沿用上一个

In [ ]:
# replay 评估（与 02 篇相同的规则）
remaining, cost_total, conv_total, alpha = BUDGET, 0.0, 0.0, 80.0
history_win, history_price = [], []
for tick, g in df.sort_values('timeStepIndex').groupby('timeStepIndex'):
    if remaining < 0.1: break
    state = dict(tick=tick, time_ratio_remaining=(NUM_TICK-tick)/NUM_TICK,
                 remaining_budget=round(remaining,1), remaining_ratio=round(remaining/BUDGET,3),
                 target_cpa=TARGET_CPA, cpa=round(cost_total/max(conv_total,1e-9),1),
                 recent_win_rate=round(np.mean(history_win[-3:]),3) if history_win else 0,
                 recent_market_price=round(np.mean(history_price[-3:]),4) if history_price else None)
    alpha = llm_alpha(state, alpha)
    bids = alpha * g.pValue.values
    win = bids >= g.leastWinningCost.values
    tick_cost = min(g.leastWinningCost.values[win].sum(), remaining)
    remaining -= tick_cost; cost_total += tick_cost
    conv_total += g.pValue.values[win].sum()
    history_win.append(win.mean()); history_price.append(g.leastWinningCost.mean())

cpa = cost_total/max(conv_total,1e-9)
score = conv_total if cpa <= TARGET_CPA else conv_total*(TARGET_CPA/cpa)**2
print(f'score={score:.2f} conv={conv_total:.1f} cpa={cpa:.1f} util={cost_total/BUDGET:.0%}')

## 练习

1. 改 PROMPT：把 pacing 规则删掉再跑——分数掉多少？（我们平台实测：这一句话值 14% 的分）
2. 把 mock 换成真 LLM（`USE_REAL_LLM=True`），观察它每步的 reasoning；
3. 给 state 加一个字段（比如最近 3 个时段各自的花费），分数会变吗？——这就是 observation 工程；
4. 终极练习：把你 02-05 实现的四个策略在同一份数据上排名，和平台排行榜
   （IQL > LLM > PID > DT）一致吗？